In [ ]:
import os
from eco2ai import Tracker
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
import torch

# --- 1. Define Prompts and Model Lists ---

# Example texts for various tasks
EXAMPLE_YELP_REVIEW = "The food was absolutely wonderful, from preparation to presentation, very pleasing..."
EXAMPLE_FRENCH_TEXT = "L'intelligence artificielle est un domaine de l'informatique qui vise à créer..."
EXAMPLE_STORY_STARTER = "The old lighthouse keeper found the strange, glowing shell on the beach..."
EXAMPLE_COT_REASONING = "A farmer has 150 meters of fencing to build a rectangular enclosure..."
EXAMPLE_NLI_PREMISE = "A group of musicians are playing various instruments on a stage..."
EXAMPLE_NLI_HYPOTHESIS = "People are performing music for a crowd."
EXAMPLE_PYTHON_FUNCTION = """
def calculate_factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    elif n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)
"""

# Library of prompts for different tasks
PROMPT_LIBRARY = {
    "binary_sentiment_analysis": f"Classify sentiment: '{EXAMPLE_YELP_REVIEW}' ->",
    "translation_fr_en": f"Translate French to English: '{EXAMPLE_FRENCH_TEXT}' ->",
    "content_creation_story": f"Write a short story starting with: '{EXAMPLE_STORY_STARTER}'",
    "chain_of_thought_reasoning": f"Solve step by step: {EXAMPLE_COT_REASONING}",
    "nli_task": f"Premise: '{EXAMPLE_NLI_PREMISE}' Hypothesis: '{EXAMPLE_NLI_HYPOTHESIS}'. Is it entailment, contradiction, or neutral?",
    "code_summarization_python": f"Summarize:\n```python\n{EXAMPLE_PYTHON_FUNCTION}\n```",
}

# Dictionary mapping tasks to the models that will be tested for that task
TASK_MODELS = {
     "chain_of_thought_reasoning": ["cot-t5-reasoning"],
}

def track_single_model_inference(task_name, model_folder, prompt):
    """
    Loads a specific local Hugging Face model, runs inference for a given prompt,
    and tracks its CO2 emissions using eco2ai.
    """
    # --- A. Define Paths ---
    base_dir = os.path.join(os.path.expanduser("~"), "GlucoseMonitoring", "uma", "models")
    model_path = os.path.join(base_dir, model_folder)

    print(f"Loading model from: {model_path}")

    # Check if the model directory exists
    if not os.path.isdir(model_path):
        print(f"Error: Model directory not found at {model_path}")
        print("Please ensure the model is correctly placed. Skipping this model.")
        return

    # --- B. Initialize the Tracker ---
    tracker = Tracker(
        project_name=f"{task_name.capitalize()}-Task",
        experiment_description=f"Inference with {model_folder}",
        file_name="model_inference_emissions.csv",
    )

    # --- C. Run and Track the Inference ---
    try:
        tracker.start()

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")

        # Wrap model/tokenizer loading in a try-except block to handle errors
        # with corrupted or incomplete model directories.
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            model = None
            
            # Try to load as a Causal LM first, then Seq2Seq, then Classification.
            try:
                model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
                print("Info: Loaded as a Causal Language Model.")
            except ValueError:
                try:
                    model = AutoModelForSeq2SeqLM.from_pretrained(model_path, device_map="auto")
                    print("Info: Loaded as a Sequence-to-Sequence Model.")
                except ValueError:
                    try:
                        model = AutoModelForSequenceClassification.from_pretrained(model_path, device_map="auto")
                        print("Info: Loaded as a Sequence Classification Model.")
                    except ValueError as e:
                         # Re-raise the error if all attempts fail
                        raise e

        except Exception as e:
            print(f"\n--- ERROR ---")
            print(f"Failed to load model or tokenizer for '{model_folder}'.")
            print(f"The model directory might be missing files or is of an unsupported type. Error: {e}")
            print("Skipping this model.")
            print("-------------\n")
            tracker.stop()
            return

        # Some models don't have a padding token defined.
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            if model.config.pad_token_id is None:
                 model.config.pad_token_id = model.config.eos_token_id

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Handle different inference types based on model architecture
        if hasattr(model, 'generate'):
            print("\nGenerating text...")
            outputs = model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        else:
            print("\nPerforming classification...")
            with torch.no_grad():
                logits = model(**inputs).logits
            predicted_class_id = logits.argmax().item()
            generated_text = model.config.id2label[predicted_class_id]

    finally:
        tracker.stop()

    # --- D. Output the Results ---
    print("\n--- Model Output ---")
    print(generated_text)
    print("----------------------\n")


def main():
    """
    Iterates through tasks and their associated models,
    running and tracking inference for each one.
    """
    # Loop through each task defined in TASK_MODELS
    for task_name, model_list in TASK_MODELS.items():
        if task_name not in PROMPT_LIBRARY:
            print(f"Warning: No prompt found for task '{task_name}'. Skipping.")
            continue
        
        # Get the corresponding prompt for the task
        prompt = PROMPT_LIBRARY[task_name]
        
        # Loop through each model assigned to that task
        for model_folder in model_list:
            print(f"\n=======================================================")
            print(f"  Running Task: {task_name} | Model: {model_folder}")
            print(f"=======================================================")
            track_single_model_inference(task_name, model_folder, prompt)

if __name__ == "__main__":
    main()

In [ ]:
import json
import time
from eco2ai import Tracker
from openai import OpenAI  # Using the official OpenAI library

# --- 1. API Configuration (User must fill this in) ---
# NOTE: This script is now configured for the OpenRouter API.
API_KEY = "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"
API_BASE_URL = "https://openrouter.ai/api/v1"

# --- 2. Define Prompts and Model Lists ---

# Dictionary of large language models to test via API
# The keys are display names, the values are the model IDs for the API provider.
API_MODELS = {
    # NOTE: You may need to find the exact model ID for your API provider.
    "Qwen": "qwen/qwen3-235b-a22b:free",
    "Mistral": "mistralai/mistral-7b-instruct",
    "DeepSeek": "deepseek/deepseek-r1-0528:free",
}

# Example texts for various tasks
EXAMPLE_YELP_REVIEW = "The food was absolutely wonderful, from preparation to presentation, very pleasing..."
EXAMPLE_FRENCH_TEXT = "L'intelligence artificielle est un domaine de l'informatique qui vise à créer..."
EXAMPLE_STORY_STARTER = "The old lighthouse keeper found the strange, glowing shell on the beach..."
EXAMPLE_COT_REASONING = "A farmer has 150 meters of fencing to build a rectangular enclosure..."
EXAMPLE_NLI_PREMISE = "A group of musicians are playing various instruments on a stage..."
EXAMPLE_NLI_HYPOTHESIS = "People are performing music for a crowd."
EXAMPLE_PYTHON_FUNCTION = """
def calculate_factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    elif n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)
"""

# Library of prompts for different tasks
PROMPT_LIBRARY = {
    "binary_sentiment_analysis": f"Classify sentiment: '{EXAMPLE_YELP_REVIEW}' ->",
    "translation_fr_en": f"Translate French to English: '{EXAMPLE_FRENCH_TEXT}' ->",
    "content_creation_story": f"Write a short story starting with: '{EXAMPLE_STORY_STARTER}'",
    "chain_of_thought_reasoning": f"Solve step by step: {EXAMPLE_COT_REASONING}",
    "nli_task": f"Premise: '{EXAMPLE_NLI_PREMISE}' Hypothesis: '{EXAMPLE_NLI_HYPOTHESIS}'. Is it entailment, contradiction, or neutral?",
    "code_summarization_python": f"Summarize:\n```python\n{EXAMPLE_PYTHON_FUNCTION}\n```",
}


def track_api_model_inference(task_name, model_name, model_id, prompt):
    """
    Calls a remote LLM via the OpenAI client, tracks the local energy usage,
    and prints the results.
    """
    print(f"Querying OpenRouter for model: {model_id}")

    # --- A. Initialize the Tracker ---
    tracker = Tracker(
        project_name=f"{task_name.capitalize()}-Task-API",
        experiment_description=f"API Inference with {model_name}",
        file_name="model_inference_emissions.csv",
    )

    # --- B. Prepare and Run the API Call ---
    # Initialize the client for each call to ensure it's fresh.
    client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    generated_text = "API call failed."

    try:
        tracker.start()
        
        # Using the client.chat.completions.create method
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.7,
            # Recommended headers can be passed as extra_headers
            extra_headers={
                "HTTP-Referer": "YOUR_SITE_URL", # Optional
                "X-Title": "YOUR_APP_NAME"      # Optional
            }
        )
        
        generated_text = response.choices[0].message.content.strip()

    except Exception as e:
        print(f"\n--- ERROR ---")
        print(f"API request failed for '{model_name}'.")
        print(f"Error Details: {e}")
        print("Please check your API_KEY and model IDs in the script.")
        print("-------------\n")
        # Add a small delay in case of rate limiting
        time.sleep(1)
    finally:
        tracker.stop()

    # --- D. Output the Results ---
    print("\n--- Model Output ---")
    print(generated_text)
    print("----------------------\n")

    


def main():
    """
    Runs and tracks inference for API-based models.
    """
    print("\n\n===== STARTING API MODEL INFERENCE =====\n")
    if API_KEY == "YOUR_OPENROUTER_API_KEY_HERE":
        print("Warning: API_KEY is not set. Skipping API model inferences.")
        return

    # Iterate through every defined task and run it on every API model
    for task_name, prompt in PROMPT_LIBRARY.items():
        for model_name, model_id in API_MODELS.items():
            print(f"\n=======================================================")
            print(f"  Running Task: {task_name} | API Model: {model_name}")
            print(f"=======================================================")
            track_api_model_inference(task_name, model_name, model_id, prompt)


if __name__ == "__main__":
    main()



In [ ]:

import os
import requests
import json
from eco2ai import Tracker
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
import torch

# --- 1. API Configuration (User must fill this in) ---
# NOTE: Replace with your actual API key and endpoint URL.
# This example uses the Together.ai API format.
API_KEY = "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"
API_URL = "https://openrouter.ai/api/v1"

# --- 2. Define Prompts and Model Lists ---

# Dictionary of large language models to test via API
# The keys are display names, the values are the model IDs for the API provider.
API_MODELS = {
    # NOTE: You may need to find the exact model ID for your API provider.
    "Qwen": "qwen/qwen3-235b-a22b:free",
    "Mistral": "mistralai/mistral-7b-instruct",
    "DeepSeek": "deepseek/deepseek-r1-0528:free",
}

# Example texts for various tasks
EXAMPLE_YELP_REVIEW = "The food was absolutely wonderful, from preparation to presentation, very pleasing..."
EXAMPLE_FRENCH_TEXT = "L'intelligence artificielle est un domaine de l'informatique qui vise à créer..."
EXAMPLE_STORY_STARTER = "The old lighthouse keeper found the strange, glowing shell on the beach..."
EXAMPLE_COT_REASONING = "A farmer has 150 meters of fencing to build a rectangular enclosure..."
EXAMPLE_NLI_PREMISE = "A group of musicians are playing various instruments on a stage..."
EXAMPLE_NLI_HYPOTHESIS = "People are performing music for a crowd."
EXAMPLE_PYTHON_FUNCTION = """
def calculate_factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers")
    elif n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)
"""

# Library of prompts for different tasks
PROMPT_LIBRARY = {
    "binary_sentiment_analysis": f"Classify sentiment: '{EXAMPLE_YELP_REVIEW}' ->",
    "translation_fr_en": f"Translate French to English: '{EXAMPLE_FRENCH_TEXT}' ->",
    "content_creation_story": f"Write a short story starting with: '{EXAMPLE_STORY_STARTER}'",
    "chain_of_thought_reasoning": f"Solve step by step: {EXAMPLE_COT_REASONING}",
    "nli_task": f"Premise: '{EXAMPLE_NLI_PREMISE}' Hypothesis: '{EXAMPLE_NLI_HYPOTHESIS}'. Is it entailment, contradiction, or neutral?",
    "code_summarization_python": f"Summarize:\n```python\n{EXAMPLE_PYTHON_FUNCTION}\n```",
}

def track_api_model_inference(task_name, model_name, model_id, prompt):
    """
    Calls a remote LLM via API, tracks the local energy usage of the call,
    and prints the results.
    """
    print(f"Querying API for model: {model_id}")

    # --- A. Initialize the Tracker ---
    tracker = Tracker(
        project_name=f"{task_name.capitalize()}-Task-API",
        experiment_description=f"API Inference with {model_name}",
        file_name="model_inference_emissions.csv",
    )

    # --- B. Prepare and Run the API Call ---
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model_id,
        "prompt": prompt,
        "max_tokens": 150,
        "temperature": 0.7,
    }
    generated_text = "API call failed."

    try:
        tracker.start()
        response = requests.post(API_URL, headers=headers, json=data)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        # --- C. Parse the Response ---
        # The response structure can vary by provider. This is a common one.
        response_data = response.json()
        if "choices" in response_data and len(response_data["choices"]) > 0:
            # For completion endpoints
            generated_text = response_data["choices"][0].get("text", "No text found in response.")
        else:
            generated_text = f"Unexpected response format: {response_data}"

    except requests.exceptions.RequestException as e:
        print(f"\n--- ERROR ---")
        print(f"API request failed for '{model_name}'. Error: {e}")
        print("Please check your API_KEY, API_URL, and model ID.")
        print("-------------\n")
    finally:
        tracker.stop()

    # --- D. Output the Results ---
    print("\n--- Model Output ---")
    print(generated_text.strip())
    print("----------------------\n")
    
def main():
   
    # --- Part 2: Run API Models ---
    print("\n\n===== STARTING API MODEL INFERENCE =====\n")
    if API_KEY == "YOUR_API_KEY_HERE":
        print("Warning: API_KEY is not set. Skipping API model inferences.")
        return

    # Iterate through every defined task and run it on every API model
    for task_name, prompt in PROMPT_LIBRARY.items():
        for model_name, model_id in API_MODELS.items():
            print(f"\n=======================================================")
            print(f"  Running Task: {task_name} | API Model: {model_name}")
            print(f"=======================================================")
            track_api_model_inference(task_name, model_name, model_id, prompt)


if __name__ == "__main__":
    main()

